# Fase 06: Enriquecimiento analítico y construcción de capa Gold

## Proyecto II Parcial - Modelado Avanzado de Base de Datos

En esta fase se construye la **capa Gold** del Data Lakehouse a partir del dataset limpio y validado de la capa Silver.

El objetivo es preparar tablas analíticas listas para consultas de negocio:

1. `gold_trips_clean`
2. `gold_daily_revenue`
3. `gold_location_performance`

Estas tablas se usarán en la siguiente fase para la carga en una base de datos consultable.

## Relación con el enunciado del proyecto

La Fase 06 solicita preparar una capa Gold orientada al análisis de negocio.

Las agregaciones obligatorias son:

- Una tabla granular de viajes limpios: `gold_trips_clean`.
- Un resumen diario de ingresos: `gold_daily_revenue`.
- Un resumen por zona de origen y destino: `gold_location_performance`.

Este notebook trabaja únicamente con Apache Spark/PySpark y mantiene la separación de capas solicitada en el proyecto: Silver como entrada y Gold como salida.

In [1]:
# Preparación de Python, Hadoop y Spark para trabajar en Windows.

import os
import sys
from pathlib import Path
from datetime import datetime

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("Python usado por Jupyter:")
print(sys.executable)

print("PYSPARK_PYTHON:")
print(os.environ["PYSPARK_PYTHON"])

print("Existe winutils:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print("Existe hadoop.dll:", os.path.exists(r"C:\hadoop\bin\hadoop.dll"))

Python usado por Jupyter:
c:\Users\Usuario\AppData\Local\Programs\Python\Python311\python.exe
PYSPARK_PYTHON:
c:\Users\Usuario\AppData\Local\Programs\Python\Python311\python.exe
Existe winutils: False
Existe hadoop.dll: False


In [2]:
# Importar librerías de Spark.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    IntegerType,
    DoubleType,
    TimestampType,
    DateType
)

In [3]:
# Crear sesión Spark.

spark = (
    SparkSession.builder
    .appName("Fase_06_Enriquecimiento_Analitico_Gold")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.files.ignoreCorruptFiles", "true")
    .config("spark.sql.files.ignoreMissingFiles", "true")
    .config("spark.hadoop.io.native.lib.available", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark iniciado correctamente.")
print("Versión Spark:", spark.version)

Spark iniciado correctamente.
Versión Spark: 4.1.2


In [4]:
# Rutas principales del proyecto.

BASE_PATH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = BASE_PATH / "data"
SILVER_PATH = DATA_PATH / "silver"
GOLD_PATH = DATA_PATH / "gold"
AUDIT_PATH = DATA_PATH / "audit"
QUARANTINE_PATH = DATA_PATH / "quarantine"

PROCESS_ID = "fase6_" + datetime.now().strftime("%Y%m%d_%H%M%S")

GOLD_PATH.mkdir(parents=True, exist_ok=True)
AUDIT_PATH.mkdir(parents=True, exist_ok=True)

print("BASE_PATH:", BASE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("AUDIT_PATH:", AUDIT_PATH)
print("PROCESS_ID:", PROCESS_ID)

BASE_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced
SILVER_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\silver
GOLD_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\gold
AUDIT_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\audit
PROCESS_ID: fase6_20260617_190427


In [5]:
# Función auxiliar para convertir rutas de Windows a formato aceptado por Spark.
# Ejemplo: C:\carpeta\archivo.parquet -> C:/carpeta/archivo.parquet

def spark_path(path):
    return str(Path(path).resolve()).replace("\\", "/")

print("Ejemplo ruta Spark:")
print(spark_path(GOLD_PATH))

Ejemplo ruta Spark:
C:/Users/Usuario/Downloads/etl_spark_parquet_advanced/etl_spark_parquet_advanced/data/gold


## 1. Lectura del dataset validado desde Silver

La Fase 06 debe partir de datos limpios y validados.  
Por eso se busca la carpeta Silver válida más reciente que contenga archivos Parquet reales.

**Importante:** no se usa `recursiveFileLookup` al leer la carpeta raíz particionada, porque Spark necesita recuperar correctamente columnas de partición como `service_type`, `year` y `month`.

In [6]:
# Buscar la carpeta Silver validada más reciente.
# Se ignoran carpetas temporales o incompletas.

silver_candidates = []

if SILVER_PATH.exists():
    for path in SILVER_PATH.glob("*"):
        if path.is_dir() and (
            path.name.startswith("validated_trips")
            or path.name.startswith("trips_validated")
            or path.name.startswith("silver_trips_valid")
        ):
            parquet_files = [
                file for file in path.rglob("*.parquet")
                if "_temporary" not in str(file)
                and not file.name.startswith(".")
                and file.name.endswith(".parquet")
            ]

            if len(parquet_files) > 0:
                silver_candidates.append({
                    "path": path,
                    "modified_time": path.stat().st_mtime,
                    "parquet_count": len(parquet_files)
                })

if len(silver_candidates) == 0:
    raise FileNotFoundError(
        "No se encontró una carpeta Silver validada con archivos .parquet reales. "
        "Primero ejecuta correctamente la Fase 05."
    )

latest_silver_info = max(silver_candidates, key=lambda item: item["modified_time"])
latest_silver_path = latest_silver_info["path"]

print("Silver validado seleccionado:")
print(latest_silver_path)
print("Archivos parquet encontrados:", latest_silver_info["parquet_count"])

Silver validado seleccionado:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\silver\validated_trips_fase5_20260617_181600
Archivos parquet encontrados: 11


In [7]:
# Leer dataset validado desde Silver.

silver_validated_trips = spark.read.parquet(spark_path(latest_silver_path))

print("Dataset Silver validado cargado correctamente.")
print("Total de registros:", silver_validated_trips.count())
print("Total de columnas:", len(silver_validated_trips.columns))

silver_validated_trips.groupBy("service_type", "year", "month").count().show(50, truncate=False)
silver_validated_trips.printSchema()
silver_validated_trips.show(5, truncate=False)

Dataset Silver validado cargado correctamente.
Total de registros: 43890529
Total de columnas: 31
+------------+----+-----+--------+
|service_type|year|month|count   |
+------------+----+-----+--------+
|yellow      |2023|1    |2991377 |
|yellow      |2023|4    |3212757 |
|yellow      |2022|2    |2920496 |
|yellow      |2023|3    |3319805 |
|yellow      |2020|1    |6280948 |
|yellow      |2021|1    |1331320 |
|fhvhv       |2023|1    |18452960|
|yellow      |2022|1    |2411744 |
|yellow      |2023|2    |2843510 |
|green       |2023|2    |61360   |
|green       |2023|1    |64252   |
+------------+----+-----+--------+

root
 |-- trip_id: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- average_speed_mph: double (nullable = tr

## 2. Lectura de métricas de calidad

La tabla `gold_daily_revenue` debe incluir `rejected_records` y `quality_percentage`.

Para ello se cargan las métricas de calidad generadas en la Fase 05, específicamente la tabla `quality_metrics_summary`.

In [8]:
# Buscar la tabla quality_metrics_summary más reciente en audit.

quality_metrics_candidates = []

if AUDIT_PATH.exists():
    for path in AUDIT_PATH.glob("*quality_metrics_summary*"):
        if path.is_dir():
            parquet_files = [
                file for file in path.rglob("*.parquet")
                if "_temporary" not in str(file)
                and not file.name.startswith(".")
            ]

            if len(parquet_files) > 0:
                quality_metrics_candidates.append({
                    "path": path,
                    "modified_time": path.stat().st_mtime,
                    "parquet_count": len(parquet_files)
                })

if len(quality_metrics_candidates) == 0:
    print("No se encontró quality_metrics_summary. Se crearán métricas por defecto para Gold.")
    quality_metrics_summary = None
else:
    latest_quality_metrics_info = max(quality_metrics_candidates, key=lambda item: item["modified_time"])
    latest_quality_metrics_path = latest_quality_metrics_info["path"]

    print("quality_metrics_summary seleccionado:")
    print(latest_quality_metrics_path)
    print("Archivos parquet encontrados:", latest_quality_metrics_info["parquet_count"])

    quality_metrics_summary = spark.read.parquet(spark_path(latest_quality_metrics_path))
    quality_metrics_summary.show(20, truncate=False)

quality_metrics_summary seleccionado:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\audit\quality_metrics_summary_fase5_20260617_181600
Archivos parquet encontrados: 1
+---------------------+------------+----+-----+-------------+-------------+----------------+-----------------+---------------------+------------------+------------------+-------------------------+
|process_id           |service_type|year|month|total_records|valid_records|rejected_records|duplicate_records|null_critical_records|suspicious_records|quality_percentage|processed_at             |
+---------------------+------------+----+-----+-------------+-------------+----------------+-----------------+---------------------+------------------+------------------+-------------------------+
|fase5_20260617_181600|yellow      |2020|1    |6281130      |6280948      |182             |0                |0                    |0                 |100.0             |2026-06-17 18:27:31.83163|
|fas

## 3. Validación de columnas requeridas para Gold

Antes de construir las tablas analíticas se verifica que el dataset Silver tenga las columnas necesarias.

Si una columna opcional no existe, se crea con valores nulos controlados.  
Esto evita que el pipeline falle por pequeñas diferencias de esquema y mantiene el proceso robusto.

In [9]:
# Columnas requeridas para la Fase 06.

required_gold_source_columns = [
    "trip_id",
    "service_type",
    "pickup_datetime",
    "dropoff_datetime",
    "trip_duration_minutes",
    "trip_distance",
    "pickup_location_id",
    "dropoff_location_id",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "tip_percentage",
    "average_speed_mph",
    "year",
    "month",
    "source_file",
    "is_suspicious_trip"
]

existing_columns = silver_validated_trips.columns
missing_columns = [col for col in required_gold_source_columns if col not in existing_columns]

print("Columnas requeridas:", len(required_gold_source_columns))
print("Columnas faltantes:", missing_columns)

gold_source = silver_validated_trips

# Crear columnas faltantes con NULL controlado si fuera necesario.
# En una ejecución correcta de Fase 04 y Fase 05, esta lista debería estar vacía.

for col_name in missing_columns:
    if col_name in ["trip_duration_minutes", "trip_distance", "fare_amount", "tip_amount", "total_amount", "tip_percentage", "average_speed_mph"]:
        gold_source = gold_source.withColumn(col_name, F.lit(None).cast(DoubleType()))
    elif col_name in ["pickup_location_id", "dropoff_location_id", "payment_type", "year", "month"]:
        gold_source = gold_source.withColumn(col_name, F.lit(None).cast(IntegerType()))
    elif col_name == "is_suspicious_trip":
        gold_source = gold_source.withColumn(col_name, F.lit(False))
    elif col_name in ["pickup_datetime", "dropoff_datetime"]:
        gold_source = gold_source.withColumn(col_name, F.lit(None).cast(TimestampType()))
    else:
        gold_source = gold_source.withColumn(col_name, F.lit(None).cast(StringType()))

print("Columnas después de control de esquema:", len(gold_source.columns))

Columnas requeridas: 18
Columnas faltantes: []
Columnas después de control de esquema: 31


## 4. Tabla Gold 1: `gold_trips_clean`

Esta tabla es granular.  
Contiene cada viaje limpio validado y solo conserva los campos analíticos solicitados en el enunciado.

In [10]:
# Construir gold_trips_clean.

gold_trips_clean = (
    gold_source
    .select(
        F.col("trip_id").cast(StringType()).alias("trip_id"),
        F.col("service_type").cast(StringType()).alias("service_type"),
        F.col("pickup_datetime").cast(TimestampType()).alias("pickup_datetime"),
        F.col("dropoff_datetime").cast(TimestampType()).alias("dropoff_datetime"),
        F.round(F.col("trip_duration_minutes").cast(DoubleType()), 2).alias("trip_duration_minutes"),
        F.round(F.col("trip_distance").cast(DoubleType()), 2).alias("trip_distance"),
        F.col("pickup_location_id").cast(IntegerType()).alias("pickup_location_id"),
        F.col("dropoff_location_id").cast(IntegerType()).alias("dropoff_location_id"),
        F.col("payment_type").cast(IntegerType()).alias("payment_type"),
        F.round(F.col("fare_amount").cast(DoubleType()), 2).alias("fare_amount"),
        F.round(F.col("tip_amount").cast(DoubleType()), 2).alias("tip_amount"),
        F.round(F.col("total_amount").cast(DoubleType()), 2).alias("total_amount"),
        F.round(F.col("tip_percentage").cast(DoubleType()), 2).alias("tip_percentage"),
        F.round(F.col("average_speed_mph").cast(DoubleType()), 2).alias("average_speed_mph"),
        F.col("year").cast(IntegerType()).alias("year"),
        F.col("month").cast(IntegerType()).alias("month"),
        F.col("source_file").cast(StringType()).alias("source_file")
    )
)

print("Tabla gold_trips_clean creada.")
print("Total de registros:", gold_trips_clean.count())
gold_trips_clean.printSchema()
gold_trips_clean.show(10, truncate=False)

Tabla gold_trips_clean creada.
Total de registros: 43890529
root
 |-- trip_id: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tip_percentage: double (nullable = true)
 |-- average_speed_mph: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- source_file: string (nullable = true)

+----------------------------------------------------------------+------------+-------------------+-------------------+---------------------+-------------+-------

## 5. Tabla Gold 2: `gold_daily_revenue`

Esta tabla resume los viajes por día y tipo de servicio.

Incluye:

- total de viajes
- ingresos totales
- tarifa promedio
- propina promedio
- distancia promedio
- duración promedio
- registros rechazados
- porcentaje de calidad

Los campos `rejected_records` y `quality_percentage` se integran desde `quality_metrics_summary`, generada en la Fase 05.

In [11]:
# Preparar métricas de calidad por servicio, año y mes.

if quality_metrics_summary is not None:
    quality_metrics_for_gold = (
        quality_metrics_summary
        .select(
            F.col("service_type").cast(StringType()).alias("qm_service_type"),
            F.col("year").cast(IntegerType()).alias("qm_year"),
            F.col("month").cast(IntegerType()).alias("qm_month"),
            F.col("rejected_records").cast(IntegerType()).alias("rejected_records"),
            F.col("quality_percentage").cast(DoubleType()).alias("quality_percentage")
        )
        .dropDuplicates(["qm_service_type", "qm_year", "qm_month"])
    )
else:
    quality_metrics_for_gold = None

# Crear resumen diario.

daily_base = (
    gold_source
    .withColumn("trip_date", F.to_date("pickup_datetime"))
    .groupBy(
        "service_type",
        "year",
        "month",
        "trip_date"
    )
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("average_fare"),
        F.round(F.avg("tip_amount"), 2).alias("average_tip"),
        F.round(F.avg("trip_distance"), 2).alias("average_trip_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("average_trip_duration")
    )
)

if quality_metrics_for_gold is not None:
    gold_daily_revenue = (
        daily_base
        .join(
            quality_metrics_for_gold,
            (daily_base["service_type"] == quality_metrics_for_gold["qm_service_type"])
            & (daily_base["year"] == quality_metrics_for_gold["qm_year"])
            & (daily_base["month"] == quality_metrics_for_gold["qm_month"]),
            "left"
        )
        .select(
            daily_base["service_type"],
            daily_base["trip_date"],
            daily_base["total_trips"],
            daily_base["total_revenue"],
            daily_base["average_fare"],
            daily_base["average_tip"],
            daily_base["average_trip_distance"],
            daily_base["average_trip_duration"],
            F.coalesce(F.col("rejected_records"), F.lit(0)).cast(IntegerType()).alias("rejected_records"),
            F.coalesce(F.col("quality_percentage"), F.lit(100.0)).cast(DoubleType()).alias("quality_percentage")
        )
    )
else:
    gold_daily_revenue = (
        daily_base
        .select(
            "service_type",
            "trip_date",
            "total_trips",
            "total_revenue",
            "average_fare",
            "average_tip",
            "average_trip_distance",
            "average_trip_duration",
            F.lit(0).cast(IntegerType()).alias("rejected_records"),
            F.lit(100.0).cast(DoubleType()).alias("quality_percentage")
        )
    )

print("Tabla gold_daily_revenue creada.")
gold_daily_revenue.printSchema()
gold_daily_revenue.orderBy("service_type", "trip_date").show(20, truncate=False)

Tabla gold_daily_revenue creada.
root
 |-- service_type: string (nullable = true)
 |-- trip_date: date (nullable = true)
 |-- total_trips: long (nullable = false)
 |-- total_revenue: double (nullable = true)
 |-- average_fare: double (nullable = true)
 |-- average_tip: double (nullable = true)
 |-- average_trip_distance: double (nullable = true)
 |-- average_trip_duration: double (nullable = true)
 |-- rejected_records: integer (nullable = false)
 |-- quality_percentage: double (nullable = false)

+------------+----------+-----------+-------------+------------+-----------+---------------------+---------------------+----------------+------------------+
|service_type|trip_date |total_trips|total_revenue|average_fare|average_tip|average_trip_distance|average_trip_duration|rejected_records|quality_percentage|
+------------+----------+-----------+-------------+------------+-----------+---------------------+---------------------+----------------+------------------+
|fhvhv       |2023-01-01|6

## 6. Tabla Gold 3: `gold_location_performance`

Esta tabla resume el rendimiento por combinación de zona de origen y zona de destino.

Permite analizar:

- rutas con más viajes
- ingresos por ruta
- tarifa promedio
- distancia promedio
- duración promedio
- cantidad de viajes sospechosos

In [12]:
# Construir gold_location_performance.

gold_location_performance = (
    gold_source
    .groupBy(
        "service_type",
        "pickup_location_id",
        "dropoff_location_id"
    )
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("average_fare"),
        F.round(F.avg("trip_distance"), 2).alias("average_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("average_duration"),
        F.sum(
            F.when(F.col("is_suspicious_trip") == True, F.lit(1)).otherwise(F.lit(0))
        ).cast(IntegerType()).alias("suspicious_trip_count")
    )
)

print("Tabla gold_location_performance creada.")
gold_location_performance.printSchema()
gold_location_performance.orderBy(F.desc("total_revenue")).show(20, truncate=False)

Tabla gold_location_performance creada.
root
 |-- service_type: string (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- total_trips: long (nullable = false)
 |-- total_revenue: double (nullable = true)
 |-- average_fare: double (nullable = true)
 |-- average_distance: double (nullable = true)
 |-- average_duration: double (nullable = true)
 |-- suspicious_trip_count: integer (nullable = true)

+------------+------------------+-------------------+-----------+-------------+------------+----------------+----------------+---------------------+
|service_type|pickup_location_id|dropoff_location_id|total_trips|total_revenue|average_fare|average_distance|average_duration|suspicious_trip_count|
+------------+------------------+-------------------+-----------+-------------+------------+----------------+----------------+---------------------+
|fhvhv       |132               |265                |65807      |6834994.16   |

## 7. Validaciones previas al guardado Gold

Antes de escribir las tablas analíticas se revisa:

- cantidad de registros
- existencia de valores nulos en campos principales
- distribución por servicio
- coherencia de ingresos

In [13]:
# Validaciones principales de las tablas Gold.

print("Registros gold_trips_clean:", gold_trips_clean.count())
print("Registros gold_daily_revenue:", gold_daily_revenue.count())
print("Registros gold_location_performance:", gold_location_performance.count())

print("Distribución de viajes limpios por servicio:")
gold_trips_clean.groupBy("service_type").count().orderBy("service_type").show(truncate=False)

print("Ingresos totales por servicio:")
gold_trips_clean.groupBy("service_type").agg(
    F.count("*").alias("total_trips"),
    F.round(F.sum("total_amount"), 2).alias("total_revenue")
).orderBy(F.desc("total_revenue")).show(truncate=False)

print("Nulos en campos críticos de gold_trips_clean:")
gold_trips_clean.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in [
        "trip_id",
        "service_type",
        "pickup_datetime",
        "dropoff_datetime",
        "trip_distance",
        "pickup_location_id",
        "dropoff_location_id",
        "total_amount",
        "year",
        "month"
    ]
]).show(truncate=False)

Registros gold_trips_clean: 43890529
Registros gold_daily_revenue: 331
Registros gold_location_performance: 108322
Distribución de viajes limpios por servicio:
+------------+--------+
|service_type|count   |
+------------+--------+
|fhvhv       |18452960|
|green       |125612  |
|yellow      |25311957|
+------------+--------+

Ingresos totales por servicio:
+------------+-----------+--------------+
|service_type|total_trips|total_revenue |
+------------+-----------+--------------+
|yellow      |25311957   |5.8781950434E8|
|fhvhv       |18452960   |4.5110300008E8|
|green       |125612     |2736537.93    |
+------------+-----------+--------------+

Nulos en campos críticos de gold_trips_clean:
+-------+------------+---------------+----------------+-------------+------------------+-------------------+------------+----+-----+
|trip_id|service_type|pickup_datetime|dropoff_datetime|trip_distance|pickup_location_id|dropoff_location_id|total_amount|year|month|
+-------+------------+-----------

## 8. Guardado de tablas analíticas en Gold

Las tres tablas se guardan en la capa `data/gold/`.

La escritura usa `mode("overwrite")` dentro de una ruta con `process_id`, por lo que cada ejecución queda separada y es auditable.

In [14]:
# Rutas de salida Gold.

gold_trips_clean_path = GOLD_PATH / f"gold_trips_clean_{PROCESS_ID}"
gold_daily_revenue_path = GOLD_PATH / f"gold_daily_revenue_{PROCESS_ID}"
gold_location_performance_path = GOLD_PATH / f"gold_location_performance_{PROCESS_ID}"

print("Ruta gold_trips_clean:", gold_trips_clean_path)
print("Ruta gold_daily_revenue:", gold_daily_revenue_path)
print("Ruta gold_location_performance:", gold_location_performance_path)

Ruta gold_trips_clean: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\gold\gold_trips_clean_fase6_20260617_190427
Ruta gold_daily_revenue: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\gold\gold_daily_revenue_fase6_20260617_190427
Ruta gold_location_performance: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\gold\gold_location_performance_fase6_20260617_190427


In [15]:
# Guardar gold_trips_clean.

(
    gold_trips_clean
    .repartition("service_type", "year", "month")
    .write
    .mode("overwrite")
    .partitionBy("service_type", "year", "month")
    .parquet(spark_path(gold_trips_clean_path))
)

print("gold_trips_clean guardada correctamente.")
print(gold_trips_clean_path)

gold_trips_clean guardada correctamente.
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\gold\gold_trips_clean_fase6_20260617_190427


In [16]:
# Guardar gold_daily_revenue.

(
    gold_daily_revenue
    .repartition("service_type")
    .write
    .mode("overwrite")
    .partitionBy("service_type")
    .parquet(spark_path(gold_daily_revenue_path))
)

print("gold_daily_revenue guardada correctamente.")
print(gold_daily_revenue_path)

gold_daily_revenue guardada correctamente.
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\gold\gold_daily_revenue_fase6_20260617_190427


In [17]:
# Guardar gold_location_performance.

(
    gold_location_performance
    .repartition("service_type")
    .write
    .mode("overwrite")
    .partitionBy("service_type")
    .parquet(spark_path(gold_location_performance_path))
)

print("gold_location_performance guardada correctamente.")
print(gold_location_performance_path)

gold_location_performance guardada correctamente.
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\gold\gold_location_performance_fase6_20260617_190427


## 9. Auditoría de generación Gold

Se genera un pequeño resumen de auditoría para dejar evidencia del número de registros creados en cada tabla Gold.

In [18]:
# Crear resumen de auditoría de la Fase 06.

audit_gold_records = [
    {
        "process_id": PROCESS_ID,
        "table_name": "gold_trips_clean",
        "output_path": str(gold_trips_clean_path),
        "record_count": gold_trips_clean.count(),
        "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    },
    {
        "process_id": PROCESS_ID,
        "table_name": "gold_daily_revenue",
        "output_path": str(gold_daily_revenue_path),
        "record_count": gold_daily_revenue.count(),
        "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    },
    {
        "process_id": PROCESS_ID,
        "table_name": "gold_location_performance",
        "output_path": str(gold_location_performance_path),
        "record_count": gold_location_performance.count(),
        "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
]

audit_gold_df = spark.createDataFrame(audit_gold_records)

audit_gold_path = AUDIT_PATH / f"audit_gold_generation_{PROCESS_ID}"

(
    audit_gold_df
    .write
    .mode("overwrite")
    .parquet(spark_path(audit_gold_path))
)

print("Auditoría de Gold guardada correctamente:")
print(audit_gold_path)

audit_gold_df.show(truncate=False)

Auditoría de Gold guardada correctamente:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\audit\audit_gold_generation_fase6_20260617_190427
+------------------------------------------------------------------------------------------------------------------------------------------+---------------------+-------------------+------------+-------------------------+
|output_path                                                                                                                               |process_id           |processed_at       |record_count|table_name               |
+------------------------------------------------------------------------------------------------------------------------------------------+---------------------+-------------------+------------+-------------------------+
|c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\gold\gold_trips_clean_fase6_20260617_190427         |fase6_20260617_1

## 10. Consultas de verificación analítica

Aunque la carga en base de datos se realiza en la Fase 07, en esta fase se hacen consultas con Spark SQL para demostrar que las tablas Gold ya son útiles para análisis.

In [19]:
# Crear vistas temporales para consultas analíticas.

gold_trips_clean.createOrReplaceTempView("gold_trips_clean")
gold_daily_revenue.createOrReplaceTempView("gold_daily_revenue")
gold_location_performance.createOrReplaceTempView("gold_location_performance")

print("Vistas temporales creadas correctamente.")

Vistas temporales creadas correctamente.


In [20]:
# Consulta 1: ingresos por servicio.

spark.sql('''
SELECT
    service_type,
    COUNT(*) AS total_trips,
    ROUND(SUM(total_amount), 2) AS total_revenue
FROM gold_trips_clean
GROUP BY service_type
ORDER BY total_revenue DESC
''').show(truncate=False)

+------------+-----------+--------------+
|service_type|total_trips|total_revenue |
+------------+-----------+--------------+
|yellow      |25311957   |5.8781950434E8|
|fhvhv       |18452960   |4.5110300008E8|
|green       |125612     |2736537.93    |
+------------+-----------+--------------+



In [21]:
# Consulta 2: ingresos diarios.

spark.sql('''
SELECT
    service_type,
    trip_date,
    total_trips,
    total_revenue,
    average_fare,
    average_tip,
    quality_percentage
FROM gold_daily_revenue
ORDER BY service_type, trip_date
''').show(30, truncate=False)

+------------+----------+-----------+-------------+------------+-----------+------------------+
|service_type|trip_date |total_trips|total_revenue|average_fare|average_tip|quality_percentage|
+------------+----------+-----------+-------------+------------+-----------+------------------+
|fhvhv       |2023-01-01|627885     |1.995907927E7|28.2        |1.0        |100.0             |
|fhvhv       |2023-01-02|407275     |1.024959778E7|22.04       |0.97       |100.0             |
|fhvhv       |2023-01-03|477183     |1.182611489E7|21.86       |0.99       |100.0             |
|fhvhv       |2023-01-04|493378     |1.218894483E7|21.83       |0.99       |100.0             |
|fhvhv       |2023-01-05|517169     |1.268918491E7|21.68       |0.98       |100.0             |
|fhvhv       |2023-01-06|607382     |1.435663004E7|20.93       |0.91       |100.0             |
|fhvhv       |2023-01-07|650106     |1.509278449E7|20.52       |0.9        |100.0             |
|fhvhv       |2023-01-08|554548     |1.3

In [22]:
# Consulta 3: rutas con mayor ingreso.

spark.sql('''
SELECT
    service_type,
    pickup_location_id,
    dropoff_location_id,
    total_trips,
    total_revenue,
    average_duration,
    suspicious_trip_count
FROM gold_location_performance
ORDER BY total_revenue DESC
LIMIT 20
''').show(truncate=False)

+------------+------------------+-------------------+-----------+-------------+----------------+---------------------+
|service_type|pickup_location_id|dropoff_location_id|total_trips|total_revenue|average_duration|suspicious_trip_count|
+------------+------------------+-------------------+-----------+-------------+----------------+---------------------+
|fhvhv       |132               |265                |65807      |6834994.16   |49.8            |0                    |
|yellow      |132               |265                |33701      |3718462.84   |39.04           |0                    |
|yellow      |132               |230                |40892      |3489886.89   |51.36           |0                    |
|fhvhv       |138               |265                |35150      |3443046.91   |43.94           |0                    |
|yellow      |264               |264                |133389     |3194786.62   |14.4            |0                    |
|yellow      |237               |236            

## Conclusión de la Fase 06

En esta fase se construyó la capa Gold del Data Lakehouse a partir del dataset validado de Silver.

Se generaron tres tablas analíticas obligatorias:

1. `gold_trips_clean`: tabla granular de viajes limpios.
2. `gold_daily_revenue`: resumen diario de ingresos por servicio.
3. `gold_location_performance`: rendimiento por zonas de origen y destino.

Estas tablas permiten analizar ingresos, cantidad de viajes, rutas principales, duración promedio, distancia promedio, propinas y calidad del dato.  
Además, se guardó una auditoría de generación Gold para mantener trazabilidad del proceso.

Con esto, los datos quedan preparados para la Fase 07: carga en base de datos consultable.